# Train a PyTorch Classifier with Minibatches

This tutorial shows how to train a small PyTorch classifier directly from Atlas
expression minibatches. Only the current expression minibatch, its labels, and
model parameters need to be available in memory.

By the end of this tutorial, you will be able to:

- define a labeled training population in `obs`;
- retrieve expression minibatches together with matching labels;
- train a neural network with randomized multi-pass minibatches;
- evaluate the training pipeline in a deterministic streaming pass;
- save a model checkpoint and feature list for later use.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- `obs.cell_type_manual` contains the target labels;
- scaled expression values are available in `data_scale`;
- the selected dense minibatches fit in memory.

Open the existing Atlas:

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import scatlaspy as sap

sap.set_progress(False)

os.chdir("/home/hanxu/scatlas-benchmarking")

atlas_path = Path("tmp/tutorials/basic_pbmc3k/pbmc3k_basic.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(
        f"Atlas database not found: {atlas_path}. Run the basic exploration tutorial first."
    )

atlas = sap.Atlas(atlas_path)

/home/hanxu/anaconda3/envs/scatlas-benchmarking/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


```{note}
Cell-type labels are used here as an example supervised target. The same
pattern can be adapted to another categorical outcome stored in `obs`.
```

## 1. Define the Labeled Training Population

Use the existing cell filter and keep only cells with a non-missing training
label. The filtered population is inspected with the `get_obs_df()` API; no
manual SQL is needed.

In [2]:
obs = atlas.get_obs_df(columns=["filter_cells", "cell_type_manual"])

training_obs = obs.loc[
    obs["filter_cells"].fillna(False) & obs["cell_type_manual"].notna()
].copy()

if training_obs.empty:
    raise ValueError("No labeled cells are available for training.")


Inspect the number of labeled cells in each class:



In [3]:
class_counts = (
    training_obs["cell_type_manual"]
    .value_counts()
    .rename_axis("cell_type_manual")
    .reset_index(name="n_cells")
)

class_counts

,cell_type_manual,n_cells
0,CD4 T,1149
1,CD14+ Monocytes,495
2,B,354
3,CD8 T,353
4,FCGR3A+ Monocytes,172
5,NK,143
6,Dendritic,34


Confirm that at least two classes are present and that each class contains
enough cells for the intended experiment.

```{important}
The training read index must select the same cell population that carries the
labels used by the model. If your atlas contains unlabeled filtered cells, add a
Boolean obs column for the labeled subset before building the read index.
```

## 2. Build the Training Read Index

Build a read index from filtered cells, filtered genes, highly variable genes,
and scaled expression values:

In [4]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)

The index table required for minibatch expression matrix reading has been rebuilt. Please rerun PCA, clustering, and other operations that depend on this table!


The model input now contains:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The trained model depends on the exact gene set, gene order, and expression
representation defined by this read index. These must be retained with the
model checkpoint.
```

## 3. Prepare the Label Encoder

Read the labels from the active read index once to fit the encoder. During
training and evaluation, labels will be retrieved directly with each minibatch
using `get_obs_col="cell_type_manual"`.

In [5]:
label_df = atlas.get_obs_df(columns=["filter_cell_id", "cell_type_manual"])
label_df = (
    label_df.dropna(subset=["filter_cell_id"])
    .sort_values("filter_cell_id")
    .reset_index(drop=True)
)


Validate the label table:



In [6]:
if label_df.empty:
    raise ValueError("The current read index contains no cells.")

if label_df["cell_type_manual"].isna().any():
    raise ValueError(
        "The current read index contains cells without labels. "
        "Create a labeled-cell obs column and rebuild the read index."
    )

if label_df["filter_cell_id"].duplicated().any():
    raise ValueError(
        "The current read index contains duplicated filter_cell_id values."
    )


Encode the class labels as consecutive integers:



In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(label_df["cell_type_manual"].to_numpy())

classes = label_encoder.classes_
n_classes = len(classes)
n_training_cells = len(label_df)

if n_classes < 2:
    raise ValueError("At least two classes are required for classification.")

print(f"Training cells: {n_training_cells:,}")
print(f"Classes: {n_classes:,}")
print(classes)

Training cells: 2,700
Classes: 7
['B' 'CD14+ Monocytes' 'CD4 T' 'CD8 T' 'Dendritic' 'FCGR3A+ Monocytes'
 'NK']


The label encoder is small and is retained in memory. The expression matrix
itself is still streamed from the Atlas database.

## 4. Configure PyTorch

Import PyTorch and select the compute device:

In [8]:
import torch
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")


Using device: cuda



Set random seeds for repeatable initialization:



In [9]:
random_seed = 42

np.random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)



```{note}
Setting random seeds improves repeatability, but exact numerical reproducibility
may still depend on the PyTorch version, hardware, and selected device
operations.
```

## 5. Define the Classifier

The number of input genes can be obtained from the first expression minibatch.
The model is therefore initialized when the first batch is encountered.

Define a small multilayer perceptron:



In [10]:
class CellTypeClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        n_classes: int,
        hidden_size: int = 128,
    ) -> None:
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        return self.network(X)



Initialize the training objects:



In [11]:
model = None
optimizer = None
loss_fn = nn.CrossEntropyLoss()

hidden_size = 128
learning_rate = 1e-3


`CrossEntropyLoss` expects unnormalized class logits from the model and integer
class identifiers from `0` to `n_classes - 1`.

## 6. Train with Label-Aware Multi-Pass Minibatches

Set the training configuration:

In [12]:
batch_size = 2048
n_epochs = 3
buffer_batch_num = 5
batches_per_epoch = int(np.ceil(n_training_cells / batch_size))

Training uses `pass_mode="multi-pass"`, which repeatedly scans the active read
index and shuffles dense minibatches through a bounded buffer. Because
`get_obs_col="cell_type_manual"` is supplied, each yielded batch contains both
`X` and the matching labels.

In [13]:
for epoch in range(1, n_epochs + 1):
    epoch_loss_sum = 0.0
    epoch_cells = 0

    if model is not None:
        model.train()

    for batch_id, batch in enumerate(
        atlas.get_minibatch_dense(
            pass_mode="multi-pass",
            batch_size=batch_size,
            buffer_batch_num=buffer_batch_num,
            max_batches=batches_per_epoch,
            get_obs_col="cell_type_manual",
        ),
        start=1,
    ):
        X_batch = np.asarray(batch["X"], dtype=np.float32)
        label_values = np.asarray(batch["cell_type_manual"], dtype=object)

        if X_batch.ndim != 2:
            raise ValueError("Each expression minibatch must be a two-dimensional matrix.")

        if pd.isna(label_values).any():
            raise ValueError("A training minibatch contains missing labels.")

        n_cells, n_genes = X_batch.shape

        if model is None:
            model = CellTypeClassifier(
                n_features=n_genes,
                n_classes=n_classes,
                hidden_size=hidden_size,
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
            model.train()

        y_batch = label_encoder.transform(label_values)

        X_tensor = torch.as_tensor(X_batch, dtype=torch.float32, device=device)
        y_tensor = torch.as_tensor(y_batch, dtype=torch.long, device=device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X_tensor)
        loss = loss_fn(logits, y_tensor)

        loss.backward()
        optimizer.step()

        epoch_loss_sum += loss.item() * n_cells
        epoch_cells += n_cells

        if batch_id == 1 or batch_id % 50 == 0:
            print(
                f"Epoch {epoch}/{n_epochs}, "
                f"batch {batch_id}: "
                f"loss={loss.item():.4f}, "
                f"processed={epoch_cells:,}"
            )

    if epoch_cells == 0:
        raise ValueError("The training stream contained no cells.")

    mean_epoch_loss = epoch_loss_sum / epoch_cells

    print(
        f"Completed epoch {epoch}/{n_epochs}: "
        f"mean loss={mean_epoch_loss:.4f}, "
        f"cells={epoch_cells:,}"
    )

Epoch 1/3, batch 1: loss=2.0520, processed=2,048
Completed epoch 1/3: mean loss=1.9922, cells=2,700
Epoch 2/3, batch 1: loss=1.5273, processed=2,048
Completed epoch 2/3: mean loss=1.4768, cells=2,700


Epoch 3/3, batch 1: loss=1.1192, processed=2,048
Completed epoch 3/3: mean loss=1.0830, cells=2,700


Only the current dense expression minibatch and its tensor representation are
required during each update.

```{note}
`multi-pass` mode should usually be used with `max_batches` so that training has
a clear stopping rule. Increasing `buffer_batch_num` improves shuffling across
nearby minibatches but increases the dense buffer memory footprint.
```

## 7. Check the Training Pipeline

Run a deterministic single-pass traversal and calculate training-set accuracy
without retaining all predictions:

In [14]:
if model is None:
    raise RuntimeError(
        "The model was not initialized because no training data were read."
    )

model.eval()

correct = 0
total = 0

with torch.inference_mode():
    for batch in atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=batch_size,
        get_obs_col="cell_type_manual",
    ):
        X_batch = np.asarray(batch["X"], dtype=np.float32)
        label_values = np.asarray(batch["cell_type_manual"], dtype=object)

        if pd.isna(label_values).any():
            raise ValueError("An evaluation minibatch contains missing labels.")

        y_batch = label_encoder.transform(label_values)
        X_tensor = torch.as_tensor(X_batch, dtype=torch.float32, device=device)

        logits = model(X_tensor)
        predictions = logits.argmax(dim=1).cpu().numpy()

        correct += int(np.count_nonzero(predictions == y_batch))
        total += X_batch.shape[0]

if total == 0:
    raise ValueError("The evaluation stream contained no cells.")

training_accuracy = correct / total

print(f"Training-set accuracy: {training_accuracy:.3f}")

Training-set accuracy: 0.906



```{warning}
Training-set accuracy is useful for checking label alignment and confirming that
the training loop functions correctly. It is not an unbiased estimate of model
performance because the same cells were used for fitting and evaluation.
```

## 8. Evaluate on Held-out Cells

For a meaningful evaluation, define non-overlapping training and test
populations in `obs`, such as:

```text
model_train_cells
model_test_cells
```

Then:

1. build the read index from `model_train_cells`;
2. retrieve the training labels in its `filter_cell_id` order;
3. train the model;
4. rebuild the read index from `model_test_cells`;
5. retrieve the test labels in the new read-index order;
6. evaluate the model with a new single-pass stream.

The labels must be queried again after each read-index construction because
`filter_cell_id` describes the current analysis view.

```{tip}
For cell-atlas applications, holding out complete donors, samples, studies, or
technologies may provide a more meaningful estimate of generalization than
randomly splitting individual cells.
```

Consider metrics such as precision, recall, F1 score, balanced accuracy, and a
confusion matrix when classes are imbalanced.

## 9. Save the Model and Input Definition

Retrieve the selected genes in read-index order:



In [15]:
feature_df = atlas.get_var_df(columns=["filter_gene_id", "atlas_gene_name"])
feature_df = (
    feature_df.dropna(subset=["filter_gene_id"])
    .sort_values("filter_gene_id")
    .reset_index()
)



Verify the feature count:



In [16]:
if len(feature_df) != model.network[0].in_features:
    raise ValueError(
        "The selected gene count does not match the model input dimension."
    )



Save the model checkpoint:



In [17]:
from pathlib import Path

output_dir = Path("tmp/tutorials/advanced/pytorch_model")
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_path = output_dir / "pytorch_cell_type_classifier.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "n_features": model.network[0].in_features,
        "hidden_size": hidden_size,
        "n_classes": n_classes,
        "classes": classes.tolist(),
        "expression_field": "data_scale",
        "use_hvg": True,
        "random_seed": random_seed,
    },
    checkpoint_path,
)

feature_df.to_csv(
    output_dir / "pytorch_model_features.csv",
    index=False,
)

print(f"Saved checkpoint to {checkpoint_path}")


Saved checkpoint to tmp/tutorials/advanced/pytorch_model/pytorch_cell_type_classifier.pt



A reusable model requires more than its learned parameters. Retain:

- the model architecture;
- class names and their encoded order;
- selected gene names and their exact order;
- the expression field;
- normalization and scaling settings;
- cell and gene selection rules;
- training hyperparameters;
- the PyTorch and scAtlasPy versions.

## 10. Load the Saved Model

Reconstruct the model using the saved architecture metadata:



In [18]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

loaded_model = CellTypeClassifier(
    n_features=checkpoint["n_features"],
    n_classes=checkpoint["n_classes"],
    hidden_size=checkpoint["hidden_size"],
).to(device)

loaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

loaded_model.eval()


CellTypeClassifier(
  (network): Sequential(
    (0): Linear(in_features=2000, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=7, bias=True)
  )
)

Before applying the loaded model, verify that the current read index contains
the same genes in the same order and uses the same expression representation as
the saved training configuration.

## Label-Aware Supervised Minibatches

When `get_obs_col` is supplied, the dense minibatch iterator returns a dictionary
containing the expression matrix and the requested obs values in matching row
order. This makes randomized supervised training possible with `multi-pass`:

```python
for batch in atlas.get_minibatch_dense(
    pass_mode="multi-pass",
    batch_size=2048,
    buffer_batch_num=5,
    max_batches=5000,
    get_obs_col="cell_type_manual",
):
    X_batch = batch["X"]
    label_values = batch["cell_type_manual"]
    train_step(X_batch, label_values)
```

Use `single-pass` when the output must follow the deterministic read-index
order, such as evaluation, prediction, or export.

## When Labels Are Not Needed

Randomized multi-pass streams can also be used when the loss does not require
labels aligned from `obs`, for example:

- autoencoder reconstruction;
- selected unsupervised representation-learning objectives;
- clustering algorithms;
- custom iterative statistics;
- methods that generate targets directly from each `X_batch`.

```python
for X_batch in atlas.get_minibatch_dense(
    pass_mode="multi-pass",
    batch_size=2048,
    buffer_batch_num=5,
    max_batches=5000,
):
    train_unsupervised_step(X_batch)
```

Always set a finite stopping condition, such as `max_batches`, for a multi-pass
training loop.


Always set a finite stopping condition, such as `max_batches`, for a multi-pass
training loop.

## Limitations of This Example

This tutorial demonstrates how Atlas minibatches can be integrated into a
PyTorch training loop. It is not intended as a complete cell-type prediction
workflow.

It does not include:

- class-imbalance correction;
- hyperparameter tuning;
- early stopping;
- regularization beyond the simple architecture;
- donor- or sample-aware validation;
- probability calibration;
- distributed or mixed-precision training.

These components should be selected according to the intended biological and
computational application.



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [19]:
atlas.close()


## Next Steps

See {doc}`apply-model-to-full-atlas` to apply the saved classifier across a
compatible Atlas read index without loading the complete expression matrix.

Continue with {doc}`implement-minibatch-kmeans` for an unsupervised iterative
method that naturally supports randomized multi-pass minibatches.